## This program creates a 3D graph of the space of nice strategies (that is a cube where the sides are p2,p3,p4) and shows the region of partner strategies in the case of stochastic games vs regular single state prisoner's dilemma 

### Initializing all parameters below

In [1]:
delta = 0.7      #Continution probability
b = 2
c = 1
R_ = b-c         # Reward R
S_ = -c          # Sucker's payoff S
T_ = b           # Temptation T
P_ = 0           # mutual defection punishment P
Q_ = 0           #0 Absorbing state payoff

P_trans_CD = 0.8   #z_2
P_trans_DD = 0.5   #z_4 

In [13]:
import pyvista as pv
import numpy as np
import itertools

# Allow empty meshes without errors
pv.global_theme.allow_empty_mesh = True



# Conditions for a nice strategy to be a partner 
def condition(p2, p3, p4, z2, z4):
    cond2 = (T_ - R_)*(1 - delta)*(1 - (1 - p3)*delta*z2) + \
            (R_ - Q_)*(1 - z2)*(delta*z2*(p2 - p3) - 1) - \
            (R_ - S_)*(1 - delta)*(1 - p2)*delta*z2 < 0
    cond3 = T_ - R_ + delta * (Q_ - T_ + (-1 + delta) * P_ * (-1 + p2) * z2 - Q_ * z2 + delta * Q_ * z2 - delta * p2 * Q_ * z2 + p2 * R_ * z2 + (-1 + p4) * (-R_ + delta * (Q_ - T_) + T_) * z4 + delta * (p2 - p4) * (Q_ - R_) * z2 * z4)<0
    return cond2 & cond3

# --- Grid setup ---
n = 500
x = np.linspace(0, 1, n)
y = np.linspace(0, 1, n)
z = np.linspace(0, 1, n)
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

# Create StructuredGrid
grid = pv.StructuredGrid(X, Y, Z)
values = condition(X, Y, Z, z2=P_trans_CD, z4=P_trans_DD).astype(float)
grid["values"] = values.ravel(order='F')

contour = grid.contour(isosurfaces=[0.5], scalars="values")

# --- Set up PyVista plotter ---
plotter = pv.Plotter(off_screen=False, window_size=[3000, 3000])

# --- Scaling factor for high-res ---
scale = 3  # tweak if needed: multiplies lines, points, and fonts

# --- Add cube edges ---
plotter.add_mesh(
    pv.Cube(bounds=(0, 1, 0, 1, 0, 1)).extract_all_edges(),
    color="black",
    line_width=2*scale
)

# --- Add smoothed surface ---
region = grid.threshold(value=0.5, scalars="values")
surface = region.extract_surface()
smoothed = surface.smooth(n_iter=50, relaxation_factor=0.1)
plotter.add_mesh(smoothed, color='#56B4E9', opacity=0.25, show_edges=False, smooth_shading=True)

# --- Feature edges ---
edges = smoothed.extract_feature_edges(
    boundary_edges=True,
    non_manifold_edges=True,
    feature_edges=True,
    feature_angle=15
)
plotter.add_mesh(edges, color="black", line_width=2*scale)

# --- REGION 2 (Regular PD) ---
new_values = condition(X, Y, Z, z2=1, z4=1).astype(float)
grid2 = pv.StructuredGrid(X, Y, Z)
grid2["new_values"] = new_values.ravel(order='F')
region2 = grid2.threshold(value=0.5, scalars="new_values")
surface2 = region2.extract_surface()
smoothed2 = surface2.smooth(n_iter=50, relaxation_factor=0.1)
plotter.add_mesh(smoothed2, color='#ff7f0e', opacity=0.8, show_edges=False)

edges2 = smoothed2.extract_feature_edges(
    boundary_edges=True,
    non_manifold_edges=True,
    feature_edges=True,
    feature_angle=15
)
plotter.add_mesh(edges2, color="black", line_width=2*scale)

# --- Cube vertices as spheres ---
cube_points = np.array(list(itertools.product([0, 1], [0, 1], [0, 1])))
plotter.add_points(
    cube_points,
    color="black",
    point_size=20*scale,
    render_points_as_spheres=True
)

# --- Strategy labels ---
strats = np.array([[1, 1, 1], [0, 1, 0], [0, 0, 1]])
names = ["ALL-C", "TFT", "WSLS"]
offset = 0.05
cube_center = np.array([0, 0, 0])
directions = strats - cube_center
label_positions = strats + offset * directions / np.linalg.norm(directions, axis=1)[:, None]

plotter.add_point_labels(
    points=label_positions,
    labels=names,
    font_size=25*scale,
    text_color="black",
    always_visible=True,
    font_family="arial",
    shape=None,
    show_points=False
)

# --- GRIM label ---
plotter.add_point_labels(
    points=cube_center - 0.09*np.array([1, 1, 1])/np.sqrt(3),
    labels=["GRIM"],
    font_size=25*scale,
    text_color="black",
    font_family="arial",
    always_visible=True,
    shape=None,
    show_points=False
)

# --- Axes lines and labels ---
L = 1.2
axes = {
    "p2": (np.array([0, 0, 0]), np.array([L, 0, 0]), 'black'),
    "p3": (np.array([0, 0, 0]), np.array([0, L, 0]), 'black'),
    "p4": (np.array([0, 0, 0]), np.array([0, 0, L]), 'black'),
}
offset_label = 0.02*scale  # scale offset too
for label, (start, end, color) in axes.items():
    plotter.add_lines(np.array([start, end]), color=color, width=2*scale)
    direction = end - start
    direction /= np.linalg.norm(direction)
    label_pos = end + offset_label * direction
    plotter.add_point_labels(
        np.array([label_pos]),
        [label],
        font_size=25*scale,
        render_points_as_spheres=False,
        fill_shape=False,
        shape=None,
        show_points=False,
        font_family="arial"
    )

plotter.show()


C:\Users\utthasani\miniconda3\Lib\site-packages\pyvista\core\utilities\points.py:77: UserWarning: Points is not a float type. This can cause issues when transforming or applying filters. Casting to ``np.float32``. Disable this by passing ``force_float=False``.
  warnings.warn(


Widget(value='<iframe src="http://localhost:52901/index.html?ui=P_0x1a7c14c3e60_7&reconnect=auto" class="pyvis…